# Word2Vec 対義語空間分析 - 実験まとめ

このノートブックでは、Word2Vecの対義語ベクトルを用いた各種実験の結果をまとめています。

## 目次
1. データ準備
2. 基本的な対義語検出
3. 多義語処理（Polysemy）
4. WSDによる文脈依存の対義語選択
5. 自動軸発見
6. 意味の加算（Semantic Arithmetic）
7. 応用（感情分析、バイアス検出等）

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from numpy.linalg import norm
import warnings
warnings.filterwarnings('ignore')

from antonym_loader import extract_antonym_pairs, group_antonyms_by_word
from word2vec_loader import Word2VecLoader

## 1. データ準備

In [ ]:
# モデルとデータの読み込み
loader = Word2VecLoader()
model = loader.load_glove(100)

antonym_pairs = extract_antonym_pairs()
known_antonyms = group_antonyms_by_word()

print(f"語彙サイズ: {len(model.key_to_index):,}")
print(f"対義語ペア数: {len(antonym_pairs):,}")
print(f"ベクトル次元: {model.vector_size}")

In [ ]:
# 対義語方向行列の構築
directions = []
direction_pairs = []

for w1, w2 in antonym_pairs:
    if w1 in model and w2 in model:
        d = model[w1] - model[w2]
        d_norm = norm(d)
        if d_norm > 1e-6:
            directions.append(d / d_norm)
            direction_pairs.append((w1, w2))

D = np.array(directions, dtype=np.float32)
print(f"有効な対義語方向: {len(D)}")

In [ ]:
# ヘルパー関数
vocab = [w for w in model.key_to_index if w.isalpha() and w == w.lower() and 3 <= len(w) <= 12][:30000]

def find_antonyms(word, top_n=10):
    """対義語を検出"""
    if word not in model:
        return []
    
    v_word = model[word]
    results = []
    
    for cand in vocab:
        if cand == word:
            continue
        v_cand = model[cand]
        diff = v_word - v_cand
        diff_norm = norm(diff)
        if diff_norm < 1e-6:
            continue
        score = np.max(np.abs(D @ (diff / diff_norm)))
        results.append((cand, score))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

def create_axis(positive, negative):
    """対義語軸を作成"""
    direction = model[positive] - model[negative]
    return direction / norm(direction)

print("ヘルパー関数を定義しました")

## 2. 基本的な対義語検出

In [ ]:
# 基本的な対義語検出のテスト
test_words = ['good', 'happy', 'hot', 'big', 'fast', 'love', 'light']

print("対義語検出結果:")
print("=" * 60)

for word in test_words:
    known = known_antonyms.get(word, set())
    detected = find_antonyms(word, top_n=5)
    
    print(f"\n{word}:")
    print(f"  既知: {known}")
    print(f"  検出: {', '.join([f'{w}({s:.2f})' for w, s in detected])}")

## 3. 多義語処理（Polysemy）

多義語は複数の対義語を持つことがあります。各意味に対して別々の軸を作成します。

In [ ]:
# 多義語の例: light
word = 'light'
light_antonyms = known_antonyms.get(word, set())
print(f"「{word}」の対義語: {light_antonyms}")

print("\n各意味での対義語検索:")
for antonym in sorted(light_antonyms):
    if antonym not in model:
        continue
    
    # この意味専用の軸
    direction = create_axis(word, antonym)
    
    # この軸上での対義語検索
    results = []
    v_word = model[word]
    for cand in vocab:
        if cand == word:
            continue
        v_cand = model[cand]
        diff = v_word - v_cand
        diff_norm = norm(diff)
        if diff_norm < 1e-6:
            continue
        alignment = abs(np.dot(diff / diff_norm, direction))
        results.append((cand, alignment))
    
    results.sort(key=lambda x: x[1], reverse=True)
    print(f"\n  {word} ↔ {antonym}:")
    print(f"    {', '.join([f'{w}({s:.2f})' for w, s in results[:5]])}")

## 4. WSDによる文脈依存の対義語選択

In [ ]:
def select_sense_by_context(word, context, antonyms):
    """文脈から適切な意味を選択"""
    context_words = [w.lower().strip('.,!?') for w in context.split() 
                    if w.lower() in model and w.lower() != word]
    
    if not context_words:
        return list(antonyms)[0] if antonyms else None
    
    context_vec = np.mean([model[w] for w in context_words], axis=0)
    
    best_antonym = None
    best_score = -float('inf')
    
    for ant in antonyms:
        if ant not in model:
            continue
        sim = np.dot(context_vec, model[ant]) / (norm(context_vec) * norm(model[ant]))
        if sim > best_score:
            best_score = sim
            best_antonym = ant
    
    return best_antonym

# テスト
test_sentences = [
    ("light", "The bag is very light and easy to carry."),
    ("light", "The room was filled with natural light."),
    ("right", "Turn right at the next intersection."),
    ("right", "That's not the right answer."),
]

print("文脈による意味選択:")
print("=" * 60)

for word, sentence in test_sentences:
    antonyms = known_antonyms.get(word, set())
    selected = select_sense_by_context(word, sentence, antonyms)
    print(f"\n\"{sentence}\"")
    print(f"  → {word} ↔ {selected}")

## 5. 自動軸発見

WordNetに頼らず、ベクトル空間から自動的に対義語軸を発見します。

In [ ]:
from sklearn.cluster import KMeans

def auto_discover_axes(word, n_axes=5, min_score=0.4):
    """自動で意味軸を発見"""
    if word not in model:
        return []
    
    v_word = model[word]
    candidates = []
    
    for cand in vocab:
        if cand == word:
            continue
        v_cand = model[cand]
        diff = v_word - v_cand
        diff_norm = norm(diff)
        if diff_norm < 1e-6 or diff_norm > 8:
            continue
        
        diff_normalized = diff / diff_norm
        score = np.max(np.abs(D @ diff_normalized))
        cosine = np.dot(v_word, v_cand) / (norm(v_word) * norm(v_cand))
        
        if score > min_score and -0.1 < cosine < 0.85:
            quality = score * (0.5 + 0.5 * max(0, cosine))
            candidates.append({
                'word': cand,
                'score': score,
                'quality': quality,
                'diff': diff_normalized
            })
    
    if len(candidates) < n_axes:
        return []
    
    candidates.sort(key=lambda x: x['quality'], reverse=True)
    candidates = candidates[:100]
    
    # クラスタリング
    diff_vectors = np.array([c['diff'] for c in candidates])
    kmeans = KMeans(n_clusters=n_axes, random_state=42, n_init=10)
    labels = kmeans.fit_predict(diff_vectors)
    
    axes = []
    for i in range(n_axes):
        cluster = [c for c, l in zip(candidates, labels) if l == i]
        if cluster:
            cluster.sort(key=lambda x: x['quality'], reverse=True)
            axes.append({
                'antonym': cluster[0]['word'],
                'score': cluster[0]['score'],
                'cluster': [c['word'] for c in cluster[:5]]
            })
    
    axes.sort(key=lambda x: x['score'], reverse=True)
    return axes

# テスト
print("自動軸発見:")
print("=" * 60)

for word in ['light', 'good', 'fast']:
    known = known_antonyms.get(word, set())
    axes = auto_discover_axes(word, n_axes=4)
    
    print(f"\n{word} (WordNet: {known}):")
    for ax in axes:
        marker = "★" if ax['antonym'] in known else "NEW"
        print(f"  [{marker}] {ax['antonym']} ({', '.join(ax['cluster'][:3])})")

## 6. 意味の加算（Semantic Arithmetic）

対義語軸を座標系として使い、意味の加算・減算を行います。

In [ ]:
def find_nearest(vector, exclude=None, top_n=10):
    """ベクトルに最も近い単語を検索"""
    exclude = set(exclude or [])
    results = []
    vec_norm = norm(vector)
    
    for w in vocab:
        if w in exclude:
            continue
        v = model[w]
        sim = np.dot(vector, v) / (vec_norm * norm(v) + 1e-10)
        results.append((w, sim))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# 強度の加算
print("強度の加算:")
print("=" * 60)

intensity_examples = [
    ('good', 'great'),
    ('bad', 'terrible'),
    ('happy', 'joyful'),
    ('cold', 'freezing'),
]

for base, strong in intensity_examples:
    intensity = model[strong] - model[base]
    v_intensified = model[base] + intensity
    neighbors = find_nearest(v_intensified, exclude=[base, strong])
    print(f"  {base} + intensity → {', '.join([w for w, _ in neighbors[:3]])}")

In [ ]:
# 属性の加算
print("\n属性の加算:")
print("=" * 60)

size_axis = create_axis('big', 'small') * 0.5
quality_axis = create_axis('good', 'bad') * 0.5

for noun in ['car', 'house', 'idea']:
    v_noun = model[noun]
    
    v_big = v_noun + size_axis * norm(model['big'] - model['small'])
    v_good = v_noun + quality_axis * norm(model['good'] - model['bad'])
    
    big_neighbors = find_nearest(v_big, exclude=[noun])
    good_neighbors = find_nearest(v_good, exclude=[noun])
    
    print(f"\n  {noun}:")
    print(f"    +big:  {', '.join([w for w, _ in big_neighbors[:3]])}")
    print(f"    +good: {', '.join([w for w, _ in good_neighbors[:3]])}")

In [ ]:
# 意味の分解
print("\n意味の分解:")
print("=" * 60)

axes = {
    'size': create_axis('big', 'small'),
    'quality': create_axis('good', 'bad'),
    'temperature': create_axis('hot', 'cold'),
    'emotion': create_axis('happy', 'sad'),
}

for word in ['excellent', 'terrible', 'cottage', 'mansion']:
    v = model[word]
    print(f"\n  {word}:")
    
    components = [(name, np.dot(v, axis)) for name, axis in axes.items()]
    components.sort(key=lambda x: abs(x[1]), reverse=True)
    
    for name, value in components[:3]:
        print(f"    {name}: {value:+.2f}")

## 7. 応用

In [ ]:
# 感情分析
print("感情分析:")
print("=" * 60)

sentiment_axis = create_axis('good', 'bad')

texts = [
    "This movie is absolutely wonderful and amazing",
    "The food was okay but nothing special",
    "That was the worst experience ever",
]

for text in texts:
    words = [w.lower().strip('.,!?') for w in text.split() if w.lower() in model]
    if words:
        vectors = [model[w] for w in words]
        avg_vec = np.mean(vectors, axis=0)
        score = np.dot(avg_vec, sentiment_axis)
        label = "Positive" if score > 0.3 else "Negative" if score < -0.3 else "Neutral"
        print(f"\n  \"{text[:50]}...\"")
        print(f"    → {label} ({score:+.2f})")

In [ ]:
# バイアス検出
print("\nバイアス検出 (性別):")
print("=" * 60)

gender_axis = create_axis('he', 'she')

occupations = ['doctor', 'nurse', 'engineer', 'teacher', 'ceo', 'secretary', 'programmer']

biases = [(w, np.dot(model[w], gender_axis)) for w in occupations if w in model]
biases.sort(key=lambda x: x[1], reverse=True)

print("\n  職業           | バイアス")
print("  " + "-" * 35)
for word, score in biases:
    direction = "Male ←" if score > 0 else "→ Female"
    print(f"  {word:15s} | {direction} ({score:+.2f})")

## まとめ

### 主な発見

1. **対義語ベクトルは意味の座標系を形成**
   - ゼロ = 中立点
   - 加算 = 軸方向への移動
   - 減算 = 意味成分の除去

2. **多義語は複数軸で表現可能**
   - light ↔ dark (明るさ)
   - light ↔ heavy (重さ)

3. **自動軸発見が可能**
   - WordNet不要で意味軸を発見
   - 新しい対義関係も発見

4. **応用可能性**
   - 感情分析
   - バイアス検出
   - 強度変換
   - 意味の分解・合成